In [2]:
import pandas as pd

# Load the cleaned movies data
movies = pd.read_csv('../data/processed/movies_clean.csv')

# Split genres into individual lists (same technique as EDA Task 5)
movies['genres_list'] = movies['genres'].str.split('|')

# Use pandas' built-in one-hot encoding helper for multi-label columns
genres_encoded = movies['genres_list'].str.join('|').str.get_dummies(sep='|')

print("Shape of genre matrix:", genres_encoded.shape)
print("\nColumns (genres found):")
print(genres_encoded.columns.tolist())
print("\nFirst 5 rows:")
genres_encoded.head()

Shape of genre matrix: (27278, 20)

Columns (genres found):
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

First 5 rows:


,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [3]:
# Drop the "(no genres listed)" column — it represents missing data, not a real genre
genres_encoded_final = genres_encoded.drop(columns=['(no genres listed)'])

print("Shape after dropping placeholder column:", genres_encoded_final.shape)
print("\nRemaining columns:")
print(genres_encoded_final.columns.tolist())

# Attach movieId so we know which row belongs to which movie
genres_encoded_final.insert(0, 'movieId', movies['movieId'])
genres_encoded_final.head()

Shape after dropping placeholder column: (27278, 19)

Remaining columns:
['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,movieId,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load cleaned tags
tags = pd.read_csv('../data/processed/tags_clean.csv')

print("Tags shape:", tags.shape)
print(f"Unique movies with at least one tag: {tags['movieId'].nunique():,}")
print(f"Out of {movies['movieId'].nunique():,} total movies in the catalog")

Tags shape: (465548, 4)
Unique movies with at least one tag: 19,545
Out of 27,278 total movies in the catalog


In [5]:
# Step 1: Combine all tags per movie into a single text "document"
# (Multiple users may have tagged the same movie with different words)
tags['tag'] = tags['tag'].astype(str)  # ensure all tags are text
movie_tags = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
movie_tags.columns = ['movieId', 'tag_text']

print("Shape of combined tags per movie:", movie_tags.shape)
print("\nExample - first 3 movies and their combined tag text:")
print(movie_tags.head(3))

Shape of combined tags per movie: (19545, 2)

Example - first 3 movies and their combined tag text:
   movieId                                           tag_text
0        1  Watched computer animation Disney animated fea...
1        2  time travel adapted from:book board game child...
2        3  old people that is actually funny sequel fever...


In [6]:
# Step 2: Convert the combined tag text into TF-IDF vectors
tfidf = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_matrix = tfidf.fit_transform(movie_tags['tag_text'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print(f"\nVocabulary size (unique meaningful words kept): {len(tfidf.vocabulary_)}")
print("\nSample of words in the vocabulary:")
print(list(tfidf.vocabulary_.keys())[:20])

TF-IDF matrix shape: (19545, 500)

Vocabulary size (unique meaningful words kept): 500

Sample of words in the vocabulary:
['watched', 'computer', 'animation', 'disney', 'animated', 'pixar', 'star', 'movie', 'family', 'tom', 'hanks', 'witty', 'adventure', 'clever', 'comedy', 'fantasy', 'humorous', 'life', 'time', 'travel']


In [8]:
# Confirm row count matches the unique-movies-with-tags number from before
print(f"movie_tags rows: {len(movie_tags):,}")
print(f"tfidf_matrix rows: {tfidf_matrix.shape[0]:,}")
print(f"Match expected count of 19,545: {tfidf_matrix.shape[0] == 19545}")

movie_tags rows: 19,545
tfidf_matrix rows: 19,545
Match expected count of 19,545: True


In [9]:
# Reload ratings and reapply the timestamp fix (fresh notebook, remember!)
ratings = pd.read_csv('../data/processed/ratings_clean.csv')

# For demonstration: take the 100 most active users and 100 most-rated movies
top_users = ratings['userId'].value_counts().head(100).index
top_movies = ratings['movieId'].value_counts().head(100).index

ratings_subset = ratings[
    ratings['userId'].isin(top_users) & ratings['movieId'].isin(top_movies)
]

print("Subset shape:", ratings_subset.shape)

# Build the user-item matrix (pivot table)
user_item_matrix_demo = ratings_subset.pivot_table(
    index='userId', columns='movieId', values='rating'
)

print("\nUser-Item matrix shape:", user_item_matrix_demo.shape)
user_item_matrix_demo.iloc[:5, :5]

Subset shape: (9278, 4)

User-Item matrix shape: (100, 100)


movieId,1,10,32,34,39
userId,,,,,
903,4.0,1.0,4.0,4.0,3.0
2261,3.0,3.0,3.5,4.0,3.0
3907,4.0,4.0,4.5,3.5,3.5
7201,4.5,3.0,3.0,5.0,3.5
8405,5.0,2.5,4.0,5.0,4.5


In [10]:
# Check how sparse even this "best case" demo matrix actually is
total_cells = user_item_matrix_demo.size
filled_cells = user_item_matrix_demo.notna().sum().sum()
empty_cells = total_cells - filled_cells

print(f"Total cells: {total_cells:,}")
print(f"Filled cells: {filled_cells:,}")
print(f"Empty cells: {empty_cells:,}")
print(f"Sparsity even among top 100 users x top 100 movies: {(empty_cells/total_cells)*100:.2f}%")

Total cells: 10,000
Filled cells: 9,278
Empty cells: 722
Sparsity even among top 100 users x top 100 movies: 7.22%


In [11]:
# Calculate each user's average rating
user_avg_rating = ratings.groupby('userId')['rating'].mean()

print("Sample of user average ratings:")
print(user_avg_rating.head(10))
print(f"\nLowest average rater: {user_avg_rating.min():.2f}")
print(f"Highest average rater: {user_avg_rating.max():.2f}")
print(f"Overall spread (max - min): {user_avg_rating.max() - user_avg_rating.min():.2f}")

Sample of user average ratings:
userId
1     3.742857
2     4.000000
3     4.122995
4     3.571429
5     4.272727
6     3.750000
7     3.289855
8     3.800000
9     3.057143
10    3.894737
Name: rating, dtype: float64

Lowest average rater: 0.50
Highest average rater: 5.00
Overall spread (max - min): 4.50


In [12]:
# Check how many ratings the most extreme average-raters actually gave
extreme_low_user = user_avg_rating.idxmin()
extreme_high_user = user_avg_rating.idxmax()

print(f"User with lowest average ({user_avg_rating.min()}): rated {(ratings['userId'] == extreme_low_user).sum()} movies total")
print(f"User with highest average ({user_avg_rating.max()}): rated {(ratings['userId'] == extreme_high_user).sum()} movies total")

User with lowest average (0.5): rated 29 movies total
User with highest average (5.0): rated 20 movies total


In [13]:
# Merge each rating with that user's average rating
ratings_with_avg = ratings.merge(
    user_avg_rating.rename('user_avg_rating'), 
    on='userId', 
    how='left'
)

# Calculate the normalized (mean-centered) rating
ratings_with_avg['rating_normalized'] = ratings_with_avg['rating'] - ratings_with_avg['user_avg_rating']

print(ratings_with_avg[['userId', 'movieId', 'rating', 'user_avg_rating', 'rating_normalized']].head(10))

print(f"\nNormalized rating range: {ratings_with_avg['rating_normalized'].min():.2f} to {ratings_with_avg['rating_normalized'].max():.2f}")
print(f"Normalized rating mean (should be very close to 0): {ratings_with_avg['rating_normalized'].mean():.4f}")

   userId  movieId  rating  user_avg_rating  rating_normalized
0       1        2     3.5         3.742857          -0.242857
1       1       29     3.5         3.742857          -0.242857
2       1       32     3.5         3.742857          -0.242857
3       1       47     3.5         3.742857          -0.242857
4       1       50     3.5         3.742857          -0.242857
5       1      112     3.5         3.742857          -0.242857
6       1      151     4.0         3.742857           0.257143
7       1      223     4.0         3.742857           0.257143
8       1      253     4.0         3.742857           0.257143
9       1      260     4.0         3.742857           0.257143

Normalized rating range: -4.39 to 4.29
Normalized rating mean (should be very close to 0): 0.0000


In [14]:
from scipy.sparse import csr_matrix

# Map userId and movieId to sequential integer positions (0, 1, 2, 3...)
# This is required because sparse matrices need clean row/column indices,
# not the actual (gapped) userId/movieId values
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

row_indices = ratings['userId'].map(user_to_idx)
col_indices = ratings['movieId'].map(movie_to_idx)

# Build the sparse matrix directly — no 27GB dense version ever created
sparse_user_item = csr_matrix(
    (ratings['rating'], (row_indices, col_indices)),
    shape=(len(user_ids), len(movie_ids))
)

print("Sparse matrix shape:", sparse_user_item.shape)
print(f"Stored (non-zero) values: {sparse_user_item.nnz:,}")
print(f"Memory used: {sparse_user_item.data.nbytes / 1e6:.2f} MB")

# Compare to what a dense version WOULD have needed
dense_memory_estimate = (len(user_ids) * len(movie_ids) * 8) / 1e9
print(f"Estimated memory if this were a DENSE matrix: {dense_memory_estimate:.2f} GB")

Sparse matrix shape: (138493, 26744)
Stored (non-zero) values: 20,000,263
Memory used: 160.00 MB
Estimated memory if this were a DENSE matrix: 29.63 GB


In [15]:
# Reverse lookups - to translate matrix positions back to real IDs later
idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}
idx_to_movie = {idx: movie_id for movie_id, idx in movie_to_idx.items()}

print("Collaborative filtering inputs ready:")
print(f"  - sparse_user_item: {sparse_user_item.shape}")
print(f"  - user_to_idx / idx_to_user: {len(user_to_idx):,} users mapped")
print(f"  - movie_to_idx / idx_to_movie: {len(movie_to_idx):,} movies mapped")

Collaborative filtering inputs ready:
  - sparse_user_item: (138493, 26744)
  - user_to_idx / idx_to_user: 138,493 users mapped
  - movie_to_idx / idx_to_movie: 26,744 movies mapped


In [16]:
print("Content-based filtering inputs ready:")
print(f"  - genres_encoded_final: {genres_encoded_final.shape}  (movieId + 19 genre features)")
print(f"  - tfidf_matrix: {tfidf_matrix.shape}  (movies with tags x 500 word features)")
print(f"  - movie_tags['movieId']: maps tfidf_matrix rows back to real movieId")

# One important note: genres cover ALL movies, tags only cover ~71.6%
# Worth explicitly confirming this gap one more time before modeling
genres_coverage = genres_encoded_final['movieId'].nunique()
tags_coverage = movie_tags['movieId'].nunique()
print(f"\nGenre coverage: {genres_coverage:,} movies")
print(f"Tag coverage: {tags_coverage:,} movies ({tags_coverage/genres_coverage*100:.1f}%)")

Content-based filtering inputs ready:
  - genres_encoded_final: (27278, 20)  (movieId + 19 genre features)
  - tfidf_matrix: (19545, 500)  (movies with tags x 500 word features)
  - movie_tags['movieId']: maps tfidf_matrix rows back to real movieId

Genre coverage: 27,278 movies
Tag coverage: 19,545 movies (71.7%)


In [17]:
import joblib
from scipy.sparse import save_npz

# 1. Genre matrix - plain tabular data, CSV is fine
genres_encoded_final.to_csv('../data/features/genres_encoded.csv', index=False)

# 2. Sparse user-item matrix - scipy has a dedicated sparse-aware save format
save_npz('../data/features/sparse_user_item.npz', sparse_user_item)

# 3. TF-IDF matrix - also sparse (mostly zeros), same dedicated format
save_npz('../data/features/tfidf_matrix.npz', tfidf_matrix)

# 4. The ID mapping dictionaries - not tabular at all, so we use joblib 
#    (a standard tool for saving arbitrary Python objects, including 
#    fitted scikit-learn models, which you'll need again on Day 3)
joblib.dump(user_to_idx, '../data/features/user_to_idx.pkl')
joblib.dump(movie_to_idx, '../data/features/movie_to_idx.pkl')
joblib.dump(idx_to_user, '../data/features/idx_to_user.pkl')
joblib.dump(idx_to_movie, '../data/features/idx_to_movie.pkl')

# 5. movie_tags - needed to map tfidf_matrix rows back to real movieIds
movie_tags[['movieId']].to_csv('../data/features/tfidf_movieId_index.csv', index=False)

# 6. The fitted TF-IDF vectorizer itself - you'll need this exact object 
#    later if you ever want to transform NEW text the same way
joblib.dump(tfidf, '../data/features/tfidf_vectorizer.pkl')

print("All feature engineering outputs saved to data/features/")

All feature engineering outputs saved to data/features/
